In [89]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
!pip install plotly
import os
import json
!pip install groq
from groq import Groq
from dotenv import load_dotenv
warnings.filterwarnings("ignore")

In [2]:
DATA_DIR = Path("data/mimic-iv-clinical-database-demo-2.2/")

print("Data directory:", DATA_DIR.resolve())
print("Exists:", DATA_DIR.exists())

Data directory: C:\Users\Dell\AI for Smarter Patient Care\data\mimic-iv-clinical-database-demo-2.2
Exists: True


In [3]:
HOSP = DATA_DIR / "hosp"
ICU = DATA_DIR / "icu"

In [4]:
print("HOSP TABLES")
print("=" * 50)

for file in sorted(HOSP.glob("*.csv.gz")):
    print(file.name)

print("\nICU TABLES")
print("=" * 50)

for file in sorted(ICU.glob("*.csv.gz")):
    print(file.name)

HOSP TABLES
admissions.csv.gz
d_hcpcs.csv.gz
d_icd_diagnoses.csv.gz
d_icd_procedures.csv.gz
d_labitems.csv.gz
diagnoses_icd.csv.gz
drgcodes.csv.gz
emar.csv.gz
emar_detail.csv.gz
hcpcsevents.csv.gz
labevents.csv.gz
microbiologyevents.csv.gz
omr.csv.gz
patients.csv.gz
pharmacy.csv.gz
poe.csv.gz
poe_detail.csv.gz
prescriptions.csv.gz
procedures_icd.csv.gz
provider.csv.gz
services.csv.gz
transfers.csv.gz

ICU TABLES
caregiver.csv.gz
chartevents.csv.gz
d_items.csv.gz
datetimeevents.csv.gz
icustays.csv.gz
ingredientevents.csv.gz
inputevents.csv.gz
outputevents.csv.gz
procedureevents.csv.gz


In [5]:
patients = pd.read_csv(HOSP / "patients.csv.gz")
admissions = pd.read_csv(HOSP / "admissions.csv.gz")
transfers = pd.read_csv(HOSP / "transfers.csv.gz")
labevents = pd.read_csv(HOSP / "labevents.csv.gz")
emar = pd.read_csv(HOSP / "emar.csv.gz")
diagnoses = pd.read_csv(HOSP / "diagnoses_icd.csv.gz")
d_labitems = pd.read_csv(HOSP / "d_labitems.csv.gz") 

In [6]:
print("Loaded tables:")
print(f"  patients:    {len(patients)} rows")
print(f"  admissions:  {len(admissions)} rows")
print(f"  transfers:   {len(transfers)} rows")
print(f"  labevents:   {len(labevents)} rows")
print(f"  emar:        {len(emar)} rows")
print(f"  diagnoses:   {len(diagnoses)} rows")

Loaded tables:
  patients:    100 rows
  admissions:  275 rows
  transfers:   1190 rows
  labevents:   107727 rows
  emar:        35835 rows
  diagnoses:   4506 rows


In [7]:
d_labitems.shape

(1622, 4)

In [8]:
icustays = pd.read_csv(ICU / "icustays.csv.gz")
d_items = pd.read_csv(ICU / "d_items.csv.gz")
 
print("\nLoaded icu tables:")
print(f"  icustays:    {len(icustays)} rows")
print(f"  d_items:     {len(d_items)} rows")


Loaded icu tables:
  icustays:    140 rows
  d_items:     4014 rows


In [9]:
for name, df in {
    "patients": patients,
    "admissions": admissions,
    "transfers": transfers,
    "labevents": labevents,
    "emar": emar,
    "diagnoses": diagnoses,
    "icustays": icustays,
    "d_items": d_items
}.items():

    print(name.upper())
    print(df.columns.tolist())

PATIENTS
['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']
ADMISSIONS
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']
TRANSFERS
['subject_id', 'hadm_id', 'transfer_id', 'eventtype', 'careunit', 'intime', 'outtime']
LABEVENTS
['labevent_id', 'subject_id', 'hadm_id', 'specimen_id', 'itemid', 'order_provider_id', 'charttime', 'storetime', 'value', 'valuenum', 'valueuom', 'ref_range_lower', 'ref_range_upper', 'flag', 'priority', 'comments']
EMAR
['subject_id', 'hadm_id', 'emar_id', 'emar_seq', 'poe_id', 'pharmacy_id', 'enter_provider_id', 'charttime', 'medication', 'event_txt', 'scheduletime', 'storetime']
DIAGNOSES
['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version']
ICUSTAYS
['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'i

In [10]:
# Patient IDs
patient_ids = set(patients["subject_id"].dropna().unique())

# Admission IDs
admission_ids = set(admissions["hadm_id"].dropna().unique())


def check_patient_relationship(df, table_name):
    if "subject_id" not in df.columns:
        return

    ids = set(df["subject_id"].dropna().unique())
    orphan_ids = ids - patient_ids

    print(
        f"{table_name:15} | "
        f"unique patients: {len(ids):5} | "
        f"orphan patients: {len(orphan_ids):5}"
    )


def check_admission_relationship(df, table_name):
    if "hadm_id" not in df.columns:
        return

    ids = set(df["hadm_id"].dropna().unique())
    orphan_ids = ids - admission_ids

    print(
        f"{table_name:15} | "
        f"unique admissions: {len(ids):5} | "
        f"orphan admissions: {len(orphan_ids):5}"
    )


print("PATIENT RELATIONSHIPS")
print("-" * 60)

for name, df in {
    "admissions": admissions,
    "transfers": transfers,
    "labevents": labevents,
    "emar": emar,
    "diagnoses": diagnoses,
    "icustays": icustays
}.items():
    check_patient_relationship(df, name)


print("\nADMISSION RELATIONSHIPS")
print("-" * 60)

for name, df in {
    "transfers": transfers,
    "labevents": labevents,
    "emar": emar,
    "diagnoses": diagnoses,
    "icustays": icustays
}.items():
    check_admission_relationship(df, name)

PATIENT RELATIONSHIPS
------------------------------------------------------------
admissions      | unique patients:   100 | orphan patients:     0
transfers       | unique patients:   100 | orphan patients:     0
labevents       | unique patients:   100 | orphan patients:     0
emar            | unique patients:    65 | orphan patients:     0
diagnoses       | unique patients:   100 | orphan patients:     0
icustays        | unique patients:   100 | orphan patients:     0

ADMISSION RELATIONSHIPS
------------------------------------------------------------
transfers       | unique admissions:   275 | orphan admissions:     0
labevents       | unique admissions:   252 | orphan admissions:     0
emar            | unique admissions:   181 | orphan admissions:     0
diagnoses       | unique admissions:   275 | orphan admissions:     0
icustays        | unique admissions:   128 | orphan admissions:     0


Our data audit revealed 100 patients but 275 admissions. That means the average patient has 2.7 hospital visits. If we just plotted by timestamp, we'd mix unrelated visits. Our AI layer is admission-aware. We force the user to select a specific visit, preventing chronological leakage—a requirement explicitly mentioned in the challenge rubric


**Key Observations :**

- **100 patients, 275 admissions** implies an average of 2.75 visits per patient. All queries and timelines must be scoped by `hadm_id` to prevent mixing data from separate hospitalizations.
- **EMAR covers only 65 patients**. When a user requests medications for a patient without EMAR records, the system must gracefully abstain.
- **ICUSTAYS covers 128 admissions**. This provides a clear test case for unsupported queries (e.g., requesting ICU data for a non-ICU admission).

In [11]:
item_lookup = d_items[ "itemid","label", "abbreviation","category","unitname"]].drop_duplicates("itemid")
print("Unique item IDs:", item_lookup["itemid"].nunique())
display(item_lookup.head(10))

Unique item IDs: 4014


,itemid,label,abbreviation,category,unitname
0,226228,Gender,Gender,ADT,NaN
1,226545,Race,Race,ADT,NaN
2,229877,Suction events (CH),Suction events (CH),ECMO,NaN
3,229875,Oxygenator visible (CH),Oxygenator visible (CH),ECMO,NaN
4,229266,Cannula sites visually inspected (ECMO),Cannula sites visually inspected (ECMO),ECMO,NaN
5,229846,Sweep (CH),Sweep (CH),ECMO,NaN
6,229267,Emergency Equipment at bedside (ECMO),Emergency Equipment at bedside (ECMO),ECMO,NaN
7,229269,Circuit inspected for clot (ECMO),Circuit inspected for clot (ECMO),ECMO,NaN
8,229876,Pump plugged into RED outlet (CH),Pump plugged into RED outlet (CH),ECMO,NaN
9,229274,Oxygenator visible (ECMO),Oxygenator visible (ECMO),ECMO,NaN


In [12]:
# Make copies so the original loaded DataFrames remain unchanged.

patients_df = patients.copy()
admissions_df = admissions.copy()
transfers_df = transfers.copy()
labevents_df = labevents.copy()
emar_df = emar.copy()
diagnoses_df = diagnoses.copy()
icustays_df = icustays.copy()
d_items_df = d_items.copy()


# Admission timestamps
for col in [
    "admittime",
    "dischtime",
    "deathtime",
    "edregtime",
    "edouttime"
]:
    if col in admissions_df.columns:
        admissions_df[col] = pd.to_datetime(
            admissions_df[col],
            errors="coerce"
        )


# Transfer timestamps
for col in ["intime", "outtime"]:
    transfers_df[col] = pd.to_datetime(
        transfers_df[col],
        errors="coerce"
    )


# Laboratory timestamps
for col in ["charttime", "storetime"]:
    labevents_df[col] = pd.to_datetime(
        labevents_df[col],
        errors="coerce"
    )


# Medication timestamps
for col in ["charttime", "scheduletime", "storetime"]:
    emar_df[col] = pd.to_datetime(
        emar_df[col],
        errors="coerce"
    )


# ICU timestamps
for col in ["intime", "outtime"]:
    icustays_df[col] = pd.to_datetime(
        icustays_df[col],
        errors="coerce"
    )

In [13]:
temporal_checks = {}

temporal_checks["invalid_admissions"] = (
    admissions_df["dischtime"].notna()
    & admissions_df["admittime"].notna()
    & (admissions_df["dischtime"] < admissions_df["admittime"])
).sum()

temporal_checks["invalid_transfers"] = (
    transfers_df["outtime"].notna()
    & transfers_df["intime"].notna()
    & (transfers_df["outtime"] < transfers_df["intime"])
).sum()

temporal_checks["invalid_icu_stays"] = (
    icustays_df["outtime"].notna()
    & icustays_df["intime"].notna()
    & (icustays_df["outtime"] < icustays_df["intime"])
).sum()

pd.DataFrame(
    [
        {"check": key, "count": value}
        for key, value in temporal_checks.items()
    ]
)

,check,count
0,invalid_admissions,0
1,invalid_transfers,0
2,invalid_icu_stays,0


In [14]:
admission_events = admissions_df[
    [
        "subject_id",
        "hadm_id",
        "admittime",
        "admission_type",
        "admission_location"
    ]
].copy()

admission_events["stay_id"] = pd.NA
admission_events["event_time"] = admission_events["admittime"]
admission_events["event_type"] = "Admission"

admission_events["description"] = (
    admission_events["admission_type"]
    .fillna("Hospital admission")
    .astype(str)
)

admission_events["source_table"] = "admissions"
admission_events["source_id"] = admission_events["hadm_id"]

admission_events = admission_events[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "event_time",
        "event_type",
        "description",
        "source_table",
        "source_id"
    ]
]

admission_events.head()

,subject_id,hadm_id,stay_id,event_time,event_type,description,source_table,source_id
0,10004235,24181354,<NA>,2196-02-24 14:38:00,Admission,URGENT,admissions,24181354
1,10009628,25926192,<NA>,2153-09-17 17:08:00,Admission,URGENT,admissions,25926192
2,10018081,23983182,<NA>,2134-08-18 02:02:00,Admission,URGENT,admissions,23983182
3,10006053,22942076,<NA>,2111-11-13 23:39:00,Admission,URGENT,admissions,22942076
4,10031404,21606243,<NA>,2113-08-04 18:46:00,Admission,URGENT,admissions,21606243


In [15]:
transfer_events = transfers_df.copy()

transfer_events["stay_id"] = pd.NA
transfer_events["event_time"] = transfer_events["intime"]
transfer_events["event_type"] = "Transfer"

transfer_events["description"] = (
    transfer_events["eventtype"].fillna("").astype(str)
    + " — "
    + transfer_events["careunit"].fillna("Unknown location").astype(str)
)

transfer_events["source_table"] = "transfers"
transfer_events["source_id"] = transfer_events["transfer_id"]

transfer_events = transfer_events[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "event_time",
        "event_type",
        "description",
        "source_table",
        "source_id"
    ]
]

transfer_events.head()

,subject_id,hadm_id,stay_id,event_time,event_type,description,source_table,source_id
0,10009049,22995465.0,<NA>,2174-05-31 14:21:47,Transfer,discharge — Unknown location,transfers,30030230
1,10025612,23403708.0,<NA>,2125-10-03 12:25:27,Transfer,discharge — Unknown location,transfers,32533329
2,10020786,23488445.0,<NA>,2189-06-13 17:25:44,Transfer,discharge — Unknown location,transfers,37922399
3,10014078,25809882.0,<NA>,2166-08-26 14:49:42,Transfer,discharge — Unknown location,transfers,34694622
4,10039831,26924951.0,<NA>,2116-01-02 14:35:02,Transfer,discharge — Unknown location,transfers,37155928


In [16]:
icu_events = icustays_df.copy()

icu_events["event_time"] = icu_events["intime"]
icu_events["event_type"] = "ICU Stay"

icu_events["description"] = (
    "ICU admission: "
    + icu_events["first_careunit"].fillna("Unknown").astype(str)
)

icu_events["source_table"] = "icustays"
icu_events["source_id"] = icu_events["stay_id"]

icu_events = icu_events[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "event_time",
        "event_type",
        "description",
        "source_table",
        "source_id"
    ]
]

icu_events.head()

,subject_id,hadm_id,stay_id,event_time,event_type,description,source_table,source_id
0,10018328,23786647,31269608,2154-04-24 23:03:44,ICU Stay,ICU admission: Neuro Stepdown,icustays,31269608
1,10020187,24104168,37509585,2169-01-15 04:56:00,ICU Stay,ICU admission: Neuro Surgical Intensive Care U...,icustays,37509585
2,10020187,26842957,32554129,2170-02-24 18:18:46,ICU Stay,ICU admission: Neuro Intermediate,icustays,32554129
3,10012853,27882036,31338022,2176-11-26 02:34:49,ICU Stay,ICU admission: Trauma SICU (TSICU),icustays,31338022
4,10020740,25826145,32145159,2150-06-03 20:12:32,ICU Stay,ICU admission: Trauma SICU (TSICU),icustays,32145159


In [17]:
# Create the laboratory dictionary lookup

lab_item_lookup = (
    d_labitems[
        ["itemid", "label", "fluid", "category"]
    ]
    .drop_duplicates("itemid")
)

# Join laboratory records with their human-readable labels

labevents_enriched = labevents_df.merge(
    lab_item_lookup,
    on="itemid",
    how="left",
    validate="many_to_one"
)

print(f"Laboratory records: {len(labevents_enriched):,}")
print(
    f"Records with labels: "
    f"{labevents_enriched['label'].notna().sum():,}"
)

display(
    labevents_enriched[
        [
            "labevent_id",
            "subject_id",
            "hadm_id",
            "itemid",
            "label",
            "value",
            "valuenum",
            "valueuom",
            "flag"
        ]
    ].head(15)
)

Laboratory records: 107,727
Records with labels: 107,727


,labevent_id,subject_id,hadm_id,itemid,label,value,valuenum,valueuom,flag
0,172061,10014354,29600294.0,51277,RDW,15.4,15.40,%,NaN
1,172062,10014354,29600294.0,51279,Red Blood Cells,3.35,3.35,m/uL,abnormal
2,172068,10014354,29600294.0,52172,RDW-SD,49.7,49.70,fL,abnormal
3,172063,10014354,29600294.0,51301,White Blood Cells,20.3,20.30,K/uL,abnormal
4,172050,10014354,29600294.0,51249,MCHC,31.1,31.10,g/dL,abnormal
5,172044,10014354,29600294.0,51221,Hematocrit,29.6,29.60,%,abnormal
6,172045,10014354,29600294.0,51222,Hemoglobin,9.2,9.20,g/dL,abnormal
7,172057,10014354,29600294.0,51265,Platelet Count,206,206.00,K/uL,NaN
8,172051,10014354,29600294.0,51250,MCV,88,88.00,fL,NaN
9,172049,10014354,29600294.0,51248,MCH,27.5,27.50,pg,NaN


In [18]:
lab_item_lookup = (
    d_labitems[
        ["itemid", "label", "fluid", "category"]
    ]
    .drop_duplicates("itemid")
)

labevents_enriched = labevents_df.merge(
    lab_item_lookup,
    on="itemid",
    how="left",
    validate="many_to_one"
)

lab_events = labevents_enriched.copy()

lab_events["stay_id"] = pd.NA
lab_events["event_time"] = lab_events["charttime"]
lab_events["event_end_time"] = pd.NaT
lab_events["event_type"] = "Laboratory"

lab_events["description"] = (
    lab_events["label"]
    .fillna("Unknown laboratory item")
    .astype(str)
    + ": "
    + lab_events["value"]
    .fillna("Not available")
    .astype(str)
    + " "
    + lab_events["valueuom"]
    .fillna("")
    .astype(str)
)

lab_events["source_table"] = "labevents"
lab_events["source_id"] = lab_events["labevent_id"]

lab_events = lab_events[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "event_time",
        "event_end_time",
        "event_type",
        "description",
        "source_table",
        "source_id"
    ]
]

display(
    lab_events[
        lab_events["subject_id"] == 10014354
    ].head(20)
)

,subject_id,hadm_id,stay_id,event_time,event_end_time,event_type,description,source_table,source_id
0,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,RDW: 15.4 %,labevents,172061
1,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,Red Blood Cells: 3.35 m/uL,labevents,172062
2,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,RDW-SD: 49.7 fL,labevents,172068
3,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,White Blood Cells: 20.3 K/uL,labevents,172063
4,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,MCHC: 31.1 g/dL,labevents,172050
5,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,Hematocrit: 29.6 %,labevents,172044
6,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,Hemoglobin: 9.2 g/dL,labevents,172045
7,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,Platelet Count: 206 K/uL,labevents,172057
8,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,MCV: 88 fL,labevents,172051
9,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,MCH: 27.5 pg,labevents,172049


In [19]:
medication_events = emar_df.copy()

medication_events["stay_id"] = pd.NA
medication_events["event_time"] = medication_events["charttime"]
medication_events["event_type"] = "Medication"

medication_events["description"] = (
    medication_events["medication"]
    .fillna("Medication event")
    .astype(str)
)

medication_events["source_table"] = "emar"
medication_events["source_id"] = medication_events["emar_id"]

medication_events = medication_events[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "event_time",
        "event_type",
        "description",
        "source_table",
        "source_id"
    ]
]

medication_events.head()

,subject_id,hadm_id,stay_id,event_time,event_type,description,source_table,source_id
0,10005909,20199380.0,<NA>,2144-10-31 05:56:00,Medication,Magnesium Sulfate,emar,10005909-74
1,10005909,20199380.0,<NA>,2144-10-31 08:00:00,Medication,Magnesium Sulfate,emar,10005909-79
2,10008287,22168393.0,<NA>,2145-09-28 20:15:00,Medication,Potassium Chloride Replacement (Critical Care ...,emar,10008287-32
3,10010471,21322534.0,<NA>,2155-05-08 21:45:00,Medication,Metoprolol Tartrate,emar,10010471-33
4,10015272,27993466.0,<NA>,2137-06-13 08:36:00,Medication,Metoprolol Tartrate,emar,10015272-31


In [20]:
diagnosis_events = diagnoses_df.copy()

diagnosis_events["stay_id"] = pd.NA
diagnosis_events["event_time"] = pd.NaT
diagnosis_events["event_type"] = "Diagnosis"

diagnosis_events["description"] = (
    "ICD-" 
    + diagnosis_events["icd_version"].astype(str)
    + ": "
    + diagnosis_events["icd_code"].astype(str)
)

diagnosis_events["source_table"] = "diagnoses"
diagnosis_events["source_id"] = (
    diagnosis_events["hadm_id"].astype(str)
    + "_"
    + diagnosis_events["seq_num"].astype(str)
)

diagnosis_events = diagnosis_events[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "event_time",
        "event_type",
        "description",
        "source_table",
        "source_id"
    ]
]

diagnosis_events.head()

,subject_id,hadm_id,stay_id,event_time,event_type,description,source_table,source_id
0,10035185,22580999,<NA>,NaT,Diagnosis,ICD-9: 4139,diagnoses,22580999_3
1,10035185,22580999,<NA>,NaT,Diagnosis,ICD-9: V707,diagnoses,22580999_10
2,10035185,22580999,<NA>,NaT,Diagnosis,ICD-9: 41401,diagnoses,22580999_1
3,10035185,22580999,<NA>,NaT,Diagnosis,ICD-9: 3899,diagnoses,22580999_9
4,10035185,22580999,<NA>,NaT,Diagnosis,ICD-9: V8532,diagnoses,22580999_11


In [21]:
event_tables = [
    admission_events,
    transfer_events,
    icu_events,
    lab_events,
    medication_events,
    diagnosis_events
]

events_df = pd.concat(
    event_tables,
    ignore_index=True
)

events_df["event_time"] = pd.to_datetime(
    events_df["event_time"],
    errors="coerce"
)

events_df = events_df.sort_values(
    by=["subject_id", "event_time"],
    na_position="last"
).reset_index(drop=True)

print("Total unified events:", f"{len(events_df):,}")

display(events_df.head(20))

Total unified events: 149,673


,subject_id,hadm_id,stay_id,event_time,event_type,description,source_table,source_id,event_end_time
0,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,Length of Urine Collection: Not available,labevents,56,NaT
1,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,Hepatitis B Surface Antibody: Not available,labevents,25,NaT
2,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,"Cocaine, Urine: NEG",labevents,55,NaT
3,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,"Amphetamine Screen, Urine: NEG",labevents,52,NaT
4,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,"Opiate Screen, Urine: NEG",labevents,59,NaT
5,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,"Benzodiazepine Screen, Urine: NEG",labevents,54,NaT
6,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,"Methadone, Urine: NEG",labevents,58,NaT
7,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,"Barbiturate Screen, Urine: NEG",labevents,53,NaT
8,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,25-OH Vitamin D: ___ ng/mL,labevents,3,NaT
9,10000032,NaN,<NA>,2180-03-23 11:51:00,Laboratory,Platelet Smear: Not available,labevents,72,NaT


In [22]:
event_distribution = (
    events_df["event_type"]
    .value_counts()
    .rename_axis("event_type")
    .reset_index(name="count")
)

display(event_distribution)

,event_type,count
0,Laboratory,107727
1,Medication,35835
2,Diagnosis,4506
3,Transfer,1190
4,Admission,275
5,ICU Stay,140


In [23]:
selected_patient = patients_df["subject_id"].iloc[0]

patient_timeline = events_df[
    events_df["subject_id"] == selected_patient
].copy()

patient_timeline = patient_timeline.sort_values(
    "event_time",
    na_position="last"
)

print("Patient:", selected_patient)
print("Events:", len(patient_timeline))

display(patient_timeline)

Patient: 10014729
Events: 658


,subject_id,hadm_id,stay_id,event_time,event_type,description,source_table,source_id,event_end_time
51456,10014729,NaN,<NA>,2125-02-22 12:15:00,Laboratory,Glucose: ___ mg/dL,labevents,187194,NaT
51486,10014729,NaN,<NA>,2125-02-22 12:15:00,Laboratory,"Protein, Total: 7.3 g/dL",labevents,187170,NaT
51487,10014729,NaN,<NA>,2125-02-22 12:15:00,Laboratory,Amylase: 58 IU/L,labevents,187158,NaT
51488,10014729,NaN,<NA>,2125-02-22 12:15:00,Laboratory,Bicarbonate: 26 mEq/L,labevents,187161,NaT
51489,10014729,NaN,<NA>,2125-02-22 12:15:00,Laboratory,Alkaline Phosphatase: 53 IU/L,labevents,187157,NaT
...,...,...,...,...,...,...,...,...,...
52109,10014729,28889419.0,<NA>,NaT,Diagnosis,ICD-9: 9962,diagnoses,28889419_2,NaT
52110,10014729,28889419.0,<NA>,NaT,Diagnosis,ICD-9: 9010,diagnoses,28889419_1,NaT
52111,10014729,28889419.0,<NA>,NaT,Diagnosis,ICD-9: E8497,diagnoses,28889419_8,NaT
52112,10014729,28889419.0,<NA>,NaT,Diagnosis,ICD-9: V1204,diagnoses,28889419_4,NaT


In [24]:
# Add event_end_time to each event type

admission_events["event_end_time"] = admissions_df["dischtime"].values

transfer_events["event_end_time"] = transfers_df["outtime"].values

icu_events["event_end_time"] = icustays_df["outtime"].values

lab_events["event_end_time"] = pd.NaT

medication_events["event_end_time"] = pd.NaT

diagnosis_events["event_end_time"] = pd.NaT


# Rebuild the unified event table

event_tables = [
    admission_events,
    transfer_events,
    icu_events,
    lab_events,
    medication_events,
    diagnosis_events
]

events_df = pd.concat(
    event_tables,
    ignore_index=True
)

events_df["event_time"] = pd.to_datetime(
    events_df["event_time"],
    errors="coerce"
)

events_df["event_end_time"] = pd.to_datetime(
    events_df["event_end_time"],
    errors="coerce"
)

events_df = events_df[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "event_time",
        "event_end_time",
        "event_type",
        "description",
        "source_table",
        "source_id"
    ]
]

events_df = events_df.sort_values(
    ["subject_id", "event_time"],
    na_position="last"
).reset_index(drop=True)

display(events_df.head(20))

,subject_id,hadm_id,stay_id,event_time,event_end_time,event_type,description,source_table,source_id
0,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,Length of Urine Collection: Not available,labevents,56
1,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,Hepatitis B Surface Antibody: Not available,labevents,25
2,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,"Cocaine, Urine: NEG",labevents,55
3,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,"Amphetamine Screen, Urine: NEG",labevents,52
4,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,"Opiate Screen, Urine: NEG",labevents,59
5,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,"Benzodiazepine Screen, Urine: NEG",labevents,54
6,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,"Methadone, Urine: NEG",labevents,58
7,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,"Barbiturate Screen, Urine: NEG",labevents,53
8,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,25-OH Vitamin D: ___ ng/mL,labevents,3
9,10000032,NaN,<NA>,2180-03-23 11:51:00,NaT,Laboratory,Platelet Smear: Not available,labevents,72


In [25]:
lab_items_path = HOSP / "d_labitems.csv.gz"

print("d_labitems exists:", lab_items_path.exists())

d_labitems exists: True


In [26]:
d_labitems = pd.read_csv(
    lab_items_path,
    compression="gzip"
)

print(d_labitems.shape)
print(d_labitems.columns.tolist())

display(d_labitems.head())

(1622, 4)
['itemid', 'label', 'fluid', 'category']


,itemid,label,fluid,category
0,50808,Free Calcium,Blood,Blood Gas
1,50826,Tidal Volume,Blood,Blood Gas
2,50813,Lactate,Blood,Blood Gas
3,52029,% Ionized Calcium,Blood,Blood Gas
4,50801,Alveolar-arterial Gradient,Blood,Blood Gas


In [27]:
def get_patient_timeline(
    subject_id,
    hadm_id=None,
    event_types=None,
    limit=100
):
    """
    Retrieve a patient's clinical timeline from the unified event table.

    Parameters
    ----------
    subject_id : int
        Patient identifier.

    hadm_id : int, optional
        Hospital admission identifier.

    event_types : list, optional
        Restrict retrieval to selected event types.

    limit : int
        Maximum number of events returned.
    """

    result = events_df[
        events_df["subject_id"] == subject_id
    ].copy()

    if hadm_id is not None:
        result = result[
            result["hadm_id"] == hadm_id
        ]

    if event_types is not None:
        result = result[
            result["event_type"].isin(event_types)
        ]

    result = result.sort_values(
        "event_time",
        na_position="last"
    )

    return result.head(limit)

In [28]:
labs = get_patient_timeline(
    subject_id=10014354,
    event_types=["Laboratory"],
    limit=50
)

display(labs)

,subject_id,hadm_id,stay_id,event_time,event_end_time,event_type,description,source_table,source_id
37290,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,EDTA Hold: Not available,labevents,167919
37346,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Glucose: ___ mg/dL,labevents,167885
37345,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Beta-2 Microglobulin: 3.0 mg/L,labevents,167891
37344,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Reticulocyte Count, Automated: 1.8 %",labevents,167951
37343,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Reticulocyte Count, Absolute: 0.07 m/uL",labevents,167950
37342,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Hepatitis B Surface Antigen: Not available,labevents,167903
37340,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Vitamin B12: 1040 pg/mL,labevents,167916
37339,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Folate: 11.3 ng/mL,labevents,167899
37338,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Ferritin: 143 ng/mL,labevents,167898
37337,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Iron Binding Capacity, Total: 373 ug/dL",labevents,167906


In [29]:
print(lab_events.shape)

display(
    lab_events[
        lab_events["subject_id"] == 10014354
    ].head(20)
)

(107727, 9)


,subject_id,hadm_id,stay_id,event_time,event_end_time,event_type,description,source_table,source_id
0,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,RDW: 15.4 %,labevents,172061
1,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,Red Blood Cells: 3.35 m/uL,labevents,172062
2,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,RDW-SD: 49.7 fL,labevents,172068
3,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,White Blood Cells: 20.3 K/uL,labevents,172063
4,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,MCHC: 31.1 g/dL,labevents,172050
5,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,Hematocrit: 29.6 %,labevents,172044
6,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,Hemoglobin: 9.2 g/dL,labevents,172045
7,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,Platelet Count: 206 K/uL,labevents,172057
8,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,MCV: 88 fL,labevents,172051
9,10014354,29600294.0,<NA>,2148-08-16,NaT,Laboratory,MCH: 27.5 pg,labevents,172049


In [30]:
event_tables = [
    admission_events,
    transfer_events,
    icu_events,
    lab_events,
    medication_events,
    diagnosis_events
]

events_df = pd.concat(
    event_tables,
    ignore_index=True
)

events_df["event_time"] = pd.to_datetime(
    events_df["event_time"],
    errors="coerce"
)

events_df["event_end_time"] = pd.to_datetime(
    events_df["event_end_time"],
    errors="coerce"
)

events_df = events_df[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "event_time",
        "event_end_time",
        "event_type",
        "description",
        "source_table",
        "source_id"
    ]
].sort_values(
    ["subject_id", "event_time"],
    na_position="last"
).reset_index(drop=True)

print("Unified event table:", events_df.shape)

display(
    events_df[
        events_df["subject_id"] == 10014354
    ].head(20)
)

Unified event table: (149673, 9)


,subject_id,hadm_id,stay_id,event_time,event_end_time,event_type,description,source_table,source_id
37290,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,EDTA Hold: Not available,labevents,167919
37291,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Red Top Hold: Not available,labevents,167917
37292,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,EDTA Hold: Not available,labevents,167884
37293,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,EDTA Hold: Not available,labevents,167920
37294,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,EDTA Hold: Not available,labevents,167883
37295,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,EDTA Hold: Not available,labevents,167881
37296,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Red Top Hold: Not available,labevents,167918
37297,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Hepatitis B Virus Core Antibody: Not available,labevents,167904
37298,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Monocytes: 1 %,labevents,167938
37299,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Absolute Basophil Count: 0.51 K/uL,labevents,167953


## Phase 2 Clinical Evidence Retrieval

The unified event table provides a chronological and traceable representation
of patient-level clinical information.

The retrieval layer identifies the most relevant evidence for a clinical
question before the information is passed to the language model.

This separates evidence retrieval from language generation and preserves
source-level traceability for every retrieved event.

In [31]:
def get_patient_timeline(subject_id, hadm_id=None, limit=100):
    """
    Retrieve chronological clinical events for a patient.
    """

    result = events_df[
        events_df["subject_id"] == subject_id
    ].copy()

    if hadm_id is not None:
        result = result[
            result["hadm_id"] == hadm_id
        ]

    result = result.sort_values(
        "event_time",
        na_position="last"
    )

    return result.head(limit).reset_index(drop=True)

In [32]:
patient_id = 10014354

timeline = get_patient_timeline(
    subject_id=patient_id,
    
)

print(f"Patient ID: {patient_id}")
print(f"Retrieved events: {len(timeline)}")

display(timeline)

Patient ID: 10014354
Retrieved events: 100


,subject_id,hadm_id,stay_id,event_time,event_end_time,event_type,description,source_table,source_id
0,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,EDTA Hold: Not available,labevents,167919
1,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Glucose: ___ mg/dL,labevents,167885
2,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Beta-2 Microglobulin: 3.0 mg/L,labevents,167891
3,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Reticulocyte Count, Automated: 1.8 %",labevents,167951
4,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Reticulocyte Count, Absolute: 0.07 m/uL",labevents,167950
...,...,...,...,...,...,...,...,...,...
95,10014354,NaN,<NA>,2146-08-07 10:20:00,NaT,Laboratory,Absolute Neutrophil Count: 5.30 K/uL,labevents,167994
96,10014354,NaN,<NA>,2146-08-07 10:20:00,NaT,Laboratory,Platelet Smear: Not available,labevents,167985
97,10014354,NaN,<NA>,2146-08-07 10:20:00,NaT,Laboratory,Neutrophils: ___ %,labevents,167981
98,10014354,NaN,<NA>,2146-08-07 10:20:00,NaT,Laboratory,Basophils: 0 %,labevents,167966


In [33]:
def search_patient_events(
    subject_id,
    query,
    hadm_id=None,
    event_type=None,
    limit=20
):
    """
    Retrieve patient events relevant to a keyword-based clinical query.
    """

    result = events_df[
        events_df["subject_id"] == subject_id
    ].copy()

    if hadm_id is not None:
        result = result[
            result["hadm_id"] == hadm_id
        ]

    if event_type is not None:
        result = result[
            result["event_type"] == event_type
        ]

    text = (
        result["description"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    query_terms = query.lower().split()

    mask = text.apply(
        lambda x: any(term in x for term in query_terms)
    )

    result = result[mask]

    return (
        result
        .sort_values("event_time", na_position="last")
        .head(limit)
        .reset_index(drop=True)
    )

In [34]:
results = search_patient_events(
    subject_id=10014354,
    query="hemoglobin"
)

display(results)

,subject_id,hadm_id,stay_id,event_time,event_end_time,event_type,description,source_table,source_id
0,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Hemoglobin: 10.5 g/dL,labevents,167929
1,10014354,NaN,<NA>,2146-08-07 10:20:00,NaT,Laboratory,% Hemoglobin A1c: ___ %,labevents,167958
2,10014354,NaN,<NA>,2146-08-07 10:20:00,NaT,Laboratory,Hemoglobin: 10.2 g/dL,labevents,167970
3,10014354,NaN,<NA>,2146-08-21 09:55:00,NaT,Laboratory,Hemoglobin: 10.6 g/dL,labevents,168038
4,10014354,NaN,<NA>,2146-08-25 14:00:00,NaT,Laboratory,Hemoglobin: 10.2 g/dL,labevents,168087
5,10014354,NaN,<NA>,2146-09-18 09:50:00,NaT,Laboratory,Hemoglobin: 10.1 g/dL,labevents,168134
6,10014354,NaN,<NA>,2146-10-08 21:40:00,NaT,Laboratory,Hemoglobin: 9.2 g/dL,labevents,168176
7,10014354,22741225.0,<NA>,2146-10-09 06:04:00,NaT,Laboratory,Hemoglobin: 8.5 g/dL,labevents,168238
8,10014354,22741225.0,<NA>,2146-10-09 06:04:00,NaT,Laboratory,% Hemoglobin A1c: ___ %,labevents,168295
9,10014354,22741225.0,<NA>,2146-10-10 09:05:00,NaT,Laboratory,Hemoglobin: 9.6 g/dL,labevents,168314


In [35]:
def get_patient_timeline(subject_id, hadm_id=None, limit=100):

    result = events_df[
        events_df["subject_id"] == subject_id
    ].copy()

    if hadm_id is not None:
        result = result[
            result["hadm_id"] == hadm_id
        ]

    result = result.sort_values(
        "event_time",
        na_position="last"
    )

    return result.head(limit).reset_index(drop=True)

In [36]:
timeline = get_patient_timeline(
    subject_id=10014354,
    limit=50
)

print("Retrieved events:", len(timeline))

display(timeline)

Retrieved events: 50


,subject_id,hadm_id,stay_id,event_time,event_end_time,event_type,description,source_table,source_id
0,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,EDTA Hold: Not available,labevents,167919
1,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Glucose: ___ mg/dL,labevents,167885
2,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Beta-2 Microglobulin: 3.0 mg/L,labevents,167891
3,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Reticulocyte Count, Automated: 1.8 %",labevents,167951
4,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Reticulocyte Count, Absolute: 0.07 m/uL",labevents,167950
5,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Hepatitis B Surface Antigen: Not available,labevents,167903
6,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Vitamin B12: 1040 pg/mL,labevents,167916
7,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Folate: 11.3 ng/mL,labevents,167899
8,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Ferritin: 143 ng/mL,labevents,167898
9,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Iron Binding Capacity, Total: 373 ug/dL",labevents,167906


In [37]:
import pandas as pd

dataframes = {
    name: obj
    for name, obj in list(globals().items())
    if isinstance(obj, pd.DataFrame)
}

for name, df in dataframes.items():
    print(f"{name}: {df.shape}")

_: (5, 8)
__: (5, 8)
___: (5, 8)
patients: (100, 6)
admissions: (275, 16)
transfers: (1190, 7)
labevents: (107727, 16)
emar: (35835, 12)
diagnoses: (4506, 5)
d_labitems: (1622, 4)
icustays: (140, 8)
d_items: (4014, 9)
df: (140, 8)
item_lookup: (4014, 5)
patients_df: (100, 6)
admissions_df: (275, 16)
transfers_df: (1190, 7)
labevents_df: (107727, 16)
emar_df: (35835, 12)
diagnoses_df: (4506, 5)
icustays_df: (140, 8)
d_items_df: (4014, 9)
_13: (3, 2)
admission_events: (275, 9)
_14: (5, 8)
transfer_events: (1190, 9)
_15: (5, 8)
icu_events: (140, 9)
_16: (5, 8)
lab_item_lookup: (1622, 4)
labevents_enriched: (107727, 19)
lab_events: (107727, 9)
medication_events: (35835, 9)
_19: (5, 8)
diagnosis_events: (4506, 9)
_20: (5, 8)
events_df: (149673, 9)
event_distribution: (6, 2)
patient_timeline: (658, 9)
labs: (50, 9)
timeline: (50, 9)
results: (20, 9)


In [38]:
events_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149673 entries, 0 to 149672
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   subject_id      149673 non-null  int64         
 1   hadm_id         120193 non-null  float64       
 2   stay_id         140 non-null     object        
 3   event_time      145167 non-null  datetime64[ns]
 4   event_end_time  1330 non-null    datetime64[ns]
 5   event_type      149673 non-null  object        
 6   description     149673 non-null  object        
 7   source_table    149673 non-null  object        
 8   source_id       149673 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(5)
memory usage: 10.3+ MB


In [39]:
events_df["event_type"].value_counts()

event_type
Laboratory    107727
Medication     35835
Diagnosis       4506
Transfer        1190
Admission        275
ICU Stay         140
Name: count, dtype: int64

In [40]:
print("Columns:")
print(timeline.columns.tolist())

print("\nSample records:")
display(timeline.head(10))

Columns:
['subject_id', 'hadm_id', 'stay_id', 'event_time', 'event_end_time', 'event_type', 'description', 'source_table', 'source_id']

Sample records:


,subject_id,hadm_id,stay_id,event_time,event_end_time,event_type,description,source_table,source_id
0,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,EDTA Hold: Not available,labevents,167919
1,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Glucose: ___ mg/dL,labevents,167885
2,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Beta-2 Microglobulin: 3.0 mg/L,labevents,167891
3,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Reticulocyte Count, Automated: 1.8 %",labevents,167951
4,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Reticulocyte Count, Absolute: 0.07 m/uL",labevents,167950
5,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Hepatitis B Surface Antigen: Not available,labevents,167903
6,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Vitamin B12: 1040 pg/mL,labevents,167916
7,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Folate: 11.3 ng/mL,labevents,167899
8,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,Ferritin: 143 ng/mL,labevents,167898
9,10014354,NaN,<NA>,2146-07-17 11:40:00,NaT,Laboratory,"Iron Binding Capacity, Total: 373 ug/dL",labevents,167906


In [91]:
patient_info = patients_df[["subject_id", "gender", "anchor_age", "dod"]].copy()
 
# Take each patient's FIRST admission for admission-level facts
first_admissions = (
    admissions_df.sort_values("admittime")
    .groupby("subject_id")
    .first()
    .reset_index()
)
 
admission_facts = first_admissions[[
    "subject_id", "admission_location", "discharge_location",
    "insurance", "marital_status", "race", "language",
    "hospital_expire_flag",
]].copy()
 
patient_info = patient_info.merge(admission_facts, on="subject_id", how="left")
 
patient_info.to_csv("patient_info.csv", index=False)
print("Saved patient_info.csv:", patient_info.shape)
display(patient_info.head())
 

Saved patient_info.csv: (100, 11)


,subject_id,gender,anchor_age,dod,admission_location,discharge_location,insurance,marital_status,race,language,hospital_expire_flag
0,10014729,F,21,NaN,PHYSICIAN REFERRAL,HOME HEALTH CARE,Other,SINGLE,WHITE - OTHER EUROPEAN,ENGLISH,0
1,10003400,F,72,2137-09-02,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,MARRIED,BLACK/AFRICAN AMERICAN,ENGLISH,0
2,10002428,F,80,NaN,EMERGENCY ROOM,SKILLED NURSING FACILITY,Medicare,WIDOWED,WHITE,ENGLISH,0
3,10032725,F,38,2143-03-30,EMERGENCY ROOM,CHRONIC/LONG TERM ACUTE CARE,Other,SINGLE,BLACK/AFRICAN AMERICAN,ENGLISH,0
4,10027445,F,48,2146-02-09,EMERGENCY ROOM,HOME HEALTH CARE,Other,WIDOWED,WHITE,ENGLISH,0


In [92]:
KEY_VITAL_LABELS = [
    "Heart Rate",
    "Respiratory Rate",
    "O2 saturation pulseoxymetry",
    "Non Invasive Blood Pressure systolic",
    "Non Invasive Blood Pressure diastolic",
    "Temperature Fahrenheit",
]
 
CHARTEVENTS_PATH = ICU / "chartevents.csv.gz"
CHUNK_SIZE = 300_000
 
# Get itemids for just these vital labels, from d_items
vital_itemids = d_items[d_items["label"].isin(KEY_VITAL_LABELS)][["itemid", "label"]]
 
vitals_chunks = []
for chunk in pd.read_csv(CHARTEVENTS_PATH, chunksize=CHUNK_SIZE,
                          usecols=["subject_id", "itemid", "charttime", "valuenum", "valueuom"]):
    filtered = chunk[chunk["itemid"].isin(vital_itemids["itemid"])]
    if not filtered.empty:
        vitals_chunks.append(filtered)
 
vitals_df = pd.concat(vitals_chunks, ignore_index=True) if vitals_chunks else pd.DataFrame()
vitals_df = vitals_df.merge(vital_itemids, on="itemid", how="left")
vitals_df["charttime"] = pd.to_datetime(vitals_df["charttime"], errors="coerce")
 
vitals_df.to_csv("vitals_df.csv", index=False)
print("Saved vitals_df.csv:", vitals_df.shape)
display(vitals_df.head())

Saved vitals_df.csv: (61441, 6)


,subject_id,charttime,itemid,valuenum,valueuom,label
0,10005817,2132-12-16 00:00:00,220210,19.0,insp/min,Respiratory Rate
1,10005817,2132-12-16 00:00:00,220045,80.0,bpm,Heart Rate
2,10005817,2132-12-16 00:00:00,220277,95.0,%,O2 saturation pulseoxymetry
3,10005817,2132-12-16 01:00:00,220045,82.0,bpm,Heart Rate
4,10005817,2132-12-16 01:00:00,220277,96.0,%,O2 saturation pulseoxymetry


In [41]:
import re
import difflib

print("events_df:", events_df.shape)
print("labevents_enriched:", labevents_enriched.shape)

print("\nevents_df columns:")
print(events_df.columns.tolist())

print("\nlabevents_enriched columns:")
print(labevents_enriched.columns.tolist())

events_df: (149673, 9)
labevents_enriched: (107727, 19)

events_df columns:
['subject_id', 'hadm_id', 'stay_id', 'event_time', 'event_end_time', 'event_type', 'description', 'source_table', 'source_id']

labevents_enriched columns:
['labevent_id', 'subject_id', 'hadm_id', 'specimen_id', 'itemid', 'order_provider_id', 'charttime', 'storetime', 'value', 'valuenum', 'valueuom', 'ref_range_lower', 'ref_range_upper', 'flag', 'priority', 'comments', 'label', 'fluid', 'category']


In [42]:
# Phase 4 — Prepare laboratory QA data

lab_qa = labevents_enriched.copy()

lab_qa["charttime"] = pd.to_datetime(
    lab_qa["charttime"],
    errors="coerce"
)

lab_qa["valuenum"] = pd.to_numeric(
    lab_qa["valuenum"],
    errors="coerce"
)

lab_qa["label"] = (
    lab_qa["label"]
    .astype(str)
    .str.strip()
)

lab_qa["valueuom"] = (
    lab_qa["valueuom"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Laboratory QA dataset:", lab_qa.shape)

display(
    lab_qa[
        [
            "subject_id",
            "hadm_id",
            "itemid",
            "label",
            "valuenum",
            "valueuom",
            "charttime",
            "flag"
        ]
    ].head(10)
)

Laboratory QA dataset: (107727, 19)


,subject_id,hadm_id,itemid,label,valuenum,valueuom,charttime,flag
0,10014354,29600294.0,51277,RDW,15.40,%,2148-08-16,NaN
1,10014354,29600294.0,51279,Red Blood Cells,3.35,m/uL,2148-08-16,abnormal
2,10014354,29600294.0,52172,RDW-SD,49.70,fL,2148-08-16,abnormal
3,10014354,29600294.0,51301,White Blood Cells,20.30,K/uL,2148-08-16,abnormal
4,10014354,29600294.0,51249,MCHC,31.10,g/dL,2148-08-16,abnormal
5,10014354,29600294.0,51221,Hematocrit,29.60,%,2148-08-16,abnormal
6,10014354,29600294.0,51222,Hemoglobin,9.20,g/dL,2148-08-16,abnormal
7,10014354,29600294.0,51265,Platelet Count,206.00,K/uL,2148-08-16,NaN
8,10014354,29600294.0,51250,MCV,88.00,fL,2148-08-16,NaN
9,10014354,29600294.0,51248,MCH,27.50,pg,2148-08-16,NaN


In [52]:
lab_coverage = (
    lab_qa
    .groupby("label")
    .agg(
        records=("label", "size"),
        numeric_values=("valuenum", "count")
    )
    .sort_values(
        "records",
        ascending=False
    )
)

display(lab_coverage.head(30))

,records,numeric_values
label,,
Glucose,3312,3039
Potassium,3022,3019
Sodium,3007,3007
Creatinine,3003,3003
Chloride,2981,2981
Urea Nitrogen,2974,2973
Hemoglobin,2930,2928
Hematocrit,2913,2908
Bicarbonate,2863,2863


In [54]:
lab_qa_flagged = lab_qa.copy()
lab_qa_flagged.to_csv("lab_qa_flagged.csv", index=False)
events_df.to_csv("events_df.csv", index=False)

In [55]:
print(lab_qa.shape)

(107727, 19)


In [56]:
VALID_METRICS = sorted(lab_qa_flagged["label"].dropna().unique().tolist())
VALID_EVENT_TYPES = sorted(events_df["event_type"].dropna().unique().tolist())

print(f"{len(VALID_METRICS)} valid lab metrics, e.g.: {VALID_METRICS[:8]}")
print(f"Valid event types: {VALID_EVENT_TYPES}")

424 valid lab metrics, e.g.: ['% Hemoglobin A1c', '24 hr Creatinine', '25-OH Vitamin D', 'ADP', 'ARCH-1', 'Absolute Basophil Count', 'Absolute CD3 Count', 'Absolute CD4 Count']
Valid event types: ['Admission', 'Diagnosis', 'ICU Stay', 'Laboratory', 'Medication', 'Transfer']


In [57]:
def _lab_evidence(df):
    df = df.drop_duplicates(subset=["label", "charttime", "valuenum"])
    return [
        {
            "source_table": "labevents",
            "source_id": str(r.get("labevent_id", r.name)),
            "timestamp": str(r["charttime"]),
            "description": r["label"],
            "value": r["valuenum"],
            "unit": r["valueuom"],
            "quality_flag": bool(r.get("quality_flag", False)),
            "quality_flag_reason": r.get("quality_flag_reason", ""),
        }
        for _, r in df.head(10).iterrows()
    ]

def _event_evidence(df):
    df = df.drop_duplicates(subset=["description", "event_time"])
    return [
        {
            "source_table": r["source_table"],
            "source_id": str(r["source_id"]),
            "timestamp": str(r["event_time"]),
            "description": r["description"],
        }
        for _, r in df.head(10).iterrows()
    ]

In [60]:
def retrieve_lab_metric(subject_id, metric, agg):
    """
    metric MUST be an exact value from VALID_METRICS (LLM is constrained
    to this list in Phase 5 — no free text reaches this function).
    agg: "max" | "min" | "latest"
    """
    if metric not in VALID_METRICS:
        return {"status": "abstained", "evidence": [],
                "reason": f"'{metric}' is not a recognized lab metric."}

    subset = lab_qa_flagged[
        (lab_qa_flagged["subject_id"] == subject_id) &
        (lab_qa_flagged["label"] == metric)
    ]
    numeric = subset[subset["valuenum"].notna()]

    if numeric.empty:
        return {"status": "abstained", "evidence": [],
                "reason": f"No numeric records for '{metric}' for this patient."}

    # Defensive: only check quality flags if the column actually exists.
    # Prevents a KeyError if lab_qa_flagged was built before the flag
    # columns were added, or reloaded from an older CSV.
    if "quality_flag_reason" in subset.columns:
        all_implausible = subset["quality_flag_reason"].str.contains(
            "implausible_value", na=False
        ).all()
        if all_implausible:
            return {"status": "abstained", "evidence": _lab_evidence(subset),
                    "reason": f"All records for '{metric}' are flagged implausible."}

    if agg == "max":
        row = numeric.loc[numeric["valuenum"].idxmax()]
    elif agg == "min":
        row = numeric.loc[numeric["valuenum"].idxmin()]
    else:
        row = numeric.sort_values("charttime").iloc[-1]

    return {"status": "answered", "evidence": _lab_evidence(pd.DataFrame([row]))}


def retrieve_event(subject_id, event_type):
    """event_type MUST be an exact value from VALID_EVENT_TYPES."""
    if event_type not in VALID_EVENT_TYPES:
        return {"status": "abstained", "evidence": [],
                "reason": f"'{event_type}' is not a recognized event type."}

    subset = events_df[
        (events_df["subject_id"] == subject_id) &
        (events_df["event_type"] == event_type)
    ].sort_values("event_time")

    if subset.empty:
        return {"status": "abstained", "evidence": [],
                "reason": f"No {event_type} records found for this patient."}

    return {"status": "answered", "evidence": _event_evidence(subset)}

In [61]:
demo_subject = 10014354
print(retrieve_lab_metric(demo_subject, "Creatinine", "max"))
print(retrieve_event(demo_subject, "Medication"))
print(retrieve_lab_metric(demo_subject, "Not A Real Test", "max"))  # should abstain

{'status': 'answered', 'evidence': [{'source_table': 'labevents', 'source_id': '170612', 'timestamp': '2148-06-26 10:15:00', 'description': 'Creatinine', 'value': 2.2, 'unit': 'mg/dL', 'quality_flag': False, 'quality_flag_reason': ''}]}
{'status': 'answered', 'evidence': [{'source_table': 'emar', 'source_id': '10014354-17', 'timestamp': '2146-10-09 00:10:00', 'description': 'Morphine Sulfate'}, {'source_table': 'emar', 'source_id': '10014354-30', 'timestamp': '2146-10-09 01:00:00', 'description': 'Medication event'}, {'source_table': 'emar', 'source_id': '10014354-29', 'timestamp': '2146-10-09 02:00:00', 'description': 'Sodium Chloride 0.9%  Flush'}, {'source_table': 'emar', 'source_id': '10014354-18', 'timestamp': '2146-10-09 05:15:00', 'description': 'Levothyroxine Sodium'}, {'source_table': 'emar', 'source_id': '10014354-19', 'timestamp': '2146-10-09 05:43:00', 'description': 'Hydrocodone-Acetaminophen (5mg-325mg)'}, {'source_table': 'emar', 'source_id': '10014354-25', 'timestamp': 

In [63]:
lab_qa_flagged.columns.tolist()

['labevent_id',
 'subject_id',
 'hadm_id',
 'specimen_id',
 'itemid',
 'order_provider_id',
 'charttime',
 'storetime',
 'value',
 'valuenum',
 'valueuom',
 'ref_range_lower',
 'ref_range_upper',
 'flag',
 'priority',
 'comments',
 'label',
 'fluid',
 'category']

In [64]:
PLAUSIBLE_RANGES = {
    "Creatinine": (0.1, 15),
    "Hemoglobin": (2, 22),
    "White Blood Cells": (0.1, 100),
    "Platelet Count": (1, 1500),
    "Glucose": (10, 1000),
    "Sodium": (100, 180),
    "Potassium": (1.5, 9),
}

lab_qa["quality_flag"] = False
lab_qa["quality_flag_reason"] = ""

def _flag(mask, reason):
    lab_qa.loc[mask, "quality_flag"] = True
    existing = lab_qa.loc[mask, "quality_flag_reason"]
    lab_qa.loc[mask, "quality_flag_reason"] = existing.apply(
        lambda r: (r + "; " if r else "") + reason
    )

missing_unit = lab_qa["valuenum"].notna() & (lab_qa["valueuom"].str.strip() == "")
_flag(missing_unit, "missing_unit")

abnormal = lab_qa["flag"].astype(str).str.lower().eq("abnormal")
_flag(abnormal, "lab_marked_abnormal")

dup_mask = lab_qa.duplicated(
    subset=["subject_id", "itemid", "charttime", "valuenum"], keep=False
)
_flag(dup_mask, "duplicate_result")

implausible = pd.Series(False, index=lab_qa.index)
for test_name, (lo, hi) in PLAUSIBLE_RANGES.items():
    test_mask = (
        lab_qa["label"].str.contains(test_name, case=False, na=False)
        & lab_qa["valuenum"].notna()
        & ((lab_qa["valuenum"] < lo) | (lab_qa["valuenum"] > hi))
    )
    implausible |= test_mask
_flag(implausible, "implausible_value")

print(f"Flagged {lab_qa['quality_flag'].sum()} / {len(lab_qa)} lab records")

Flagged 45906 / 107727 lab records


In [65]:
lab_qa_flagged = lab_qa.copy()
lab_qa_flagged.to_csv("lab_qa_flagged.csv", index=False)
events_df.to_csv("events_df.csv", index=False)

print(lab_qa_flagged.columns.tolist())

['labevent_id', 'subject_id', 'hadm_id', 'specimen_id', 'itemid', 'order_provider_id', 'charttime', 'storetime', 'value', 'valuenum', 'valueuom', 'ref_range_lower', 'ref_range_upper', 'flag', 'priority', 'comments', 'label', 'fluid', 'category', 'quality_flag', 'quality_flag_reason']


In [66]:
print(retrieve_lab_metric(demo_subject, "Creatinine", "max"))
print(retrieve_event(demo_subject, "Medication"))
print(retrieve_lab_metric(demo_subject, "Not A Real Test", "max"))  # should abstain

{'status': 'answered', 'evidence': [{'source_table': 'labevents', 'source_id': '170612', 'timestamp': '2148-06-26 10:15:00', 'description': 'Creatinine', 'value': 2.2, 'unit': 'mg/dL', 'quality_flag': True, 'quality_flag_reason': 'lab_marked_abnormal'}]}
{'status': 'answered', 'evidence': [{'source_table': 'emar', 'source_id': '10014354-17', 'timestamp': '2146-10-09 00:10:00', 'description': 'Morphine Sulfate'}, {'source_table': 'emar', 'source_id': '10014354-30', 'timestamp': '2146-10-09 01:00:00', 'description': 'Medication event'}, {'source_table': 'emar', 'source_id': '10014354-29', 'timestamp': '2146-10-09 02:00:00', 'description': 'Sodium Chloride 0.9%  Flush'}, {'source_table': 'emar', 'source_id': '10014354-18', 'timestamp': '2146-10-09 05:15:00', 'description': 'Levothyroxine Sodium'}, {'source_table': 'emar', 'source_id': '10014354-19', 'timestamp': '2146-10-09 05:43:00', 'description': 'Hydrocodone-Acetaminophen (5mg-325mg)'}, {'source_table': 'emar', 'source_id': '10014354-

In [76]:
load_dotenv()
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.3-70b-versatile"
print("LLM client initialized.")

LLM client initialized.


In [77]:
QUERY_TOOL = {
    "type": "function",
    "function": {
        "name": "structured_query",
        "description": "Convert a clinical question into a structured lookup.",
        "parameters": {
            "type": "object",
            "properties": {
                "query_type": {"type": "string", "enum": ["lab_metric", "event_lookup", "unsupported"]},
                "metric": {"type": "string", "enum": VALID_METRICS + [""]},
                "agg": {"type": "string", "enum": ["max", "min", "latest", ""]},
                "event_type": {"type": "string", "enum": [e for e in VALID_EVENT_TYPES if e != "Laboratory"] + [""]},
            },
            "required": ["query_type"],
        },
    },
}

def parse_question_with_llm(question):
    resp = client.chat.completions.create(
        model=MODEL,
        max_tokens=300,
        tools=[QUERY_TOOL],
        tool_choice={"type": "function", "function": {"name": "structured_query"}},
        messages=[{
            "role": "user",
            "content": f"Question: {question}\n\n"
                       f"If it asks about a lab/vital numeric value, set query_type='lab_metric' "
                       f"and pick the closest matching metric from the allowed list, plus agg. "
                       f"Never use event_type='Laboratory' for numeric value questions — "
                       f"those must use query_type='lab_metric'. "
                       f"If it asks about medications/admissions/ICU/diagnoses, set "
                       f"query_type='event_lookup' and pick the matching event_type. "
                       f"If neither fits, set query_type='unsupported'."
        }],
    )
    tool_call = resp.choices[0].message.tool_calls[0]
    return json.loads(tool_call.function.arguments)

In [78]:
def generate_grounded_response(question, retrieval_result):
    if retrieval_result["status"] == "abstained" or not retrieval_result["evidence"]:
        return f"Cannot answer — {retrieval_result.get('reason', 'no supporting record found.')}"

    evidence_json = json.dumps(retrieval_result["evidence"], default=str)

    resp = client.chat.completions.create(
        model=MODEL,
        max_tokens=200,
        messages=[
            {"role": "system", "content": (
                "You answer using ONLY the facts in the evidence JSON provided. "
                "Never state a number, date, or fact not present in the evidence. "
                "If the evidence is empty, respond exactly: "
                "'Cannot answer — no supporting record found.' "
                "Always mention the source_table and timestamp for traceability."
            )},
            {"role": "user", "content": f"Question: {question}\n\nEvidence: {evidence_json}\n\nAnswer concisely."},
        ],
    )
    return resp.choices[0].message.content

In [79]:
def llm_answer_question(question, subject_id):
    query = parse_question_with_llm(question)

    if query["query_type"] == "lab_metric" and query.get("metric") and query.get("agg"):
        result = retrieve_lab_metric(subject_id, query["metric"], query["agg"])
    elif query["query_type"] == "event_lookup" and query.get("event_type"):
        result = retrieve_event(subject_id, query["event_type"])
    else:
        result = {"status": "abstained", "evidence": [],
                  "reason": "Question doesn't match a supported query type."}

    answer_text = generate_grounded_response(question, result)
    return {"query": query, "retrieval": result, "answer": answer_text}

In [80]:
test_questions = [
    "What was the highest creatinine value?",
    "What medications were given?",
    "What is the weather today?",
]
for q in test_questions:
    out = llm_answer_question(q, demo_subject)
    print(f"\nQ: {q}")
    print(f"   parsed query: {out['query']}")
    print(f"   answer: {out['answer']}")


Q: What was the highest creatinine value?
   parsed query: {'agg': 'max', 'metric': 'Creatinine', 'query_type': 'lab_metric'}
   answer: The highest creatinine value was 2.2 mg/dL (source_table: labevents, timestamp: 2148-06-26 10:15:00).

Q: What medications were given?
   parsed query: {'event_type': 'Medication', 'query_type': 'event_lookup'}
   answer: Medications given: Morphine Sulfate, Levothyroxine Sodium, Hydrocodone-Acetaminophen, Insulin, Senna, ZIPRASidone Hydrochloride, Pantoprazole, and Docusate Sodium. (Source: emar, various timestamps)

Q: What is the weather today?
   parsed query: {'query_type': 'unsupported'}
   answer: Cannot answer — Question doesn't match a supported query type.


In [93]:
import subprocess
process = subprocess.Popen(["streamlit", "run", "app.py"])
print("Streamlit started — open http://localhost:8501 in your browser")

Streamlit started — open http://localhost:8501 in your browser
